# Fine-tuning Self-Pretrained ViT on Higher Resolution Images

## Task: Binary Classification (Pizza vs Sushi) from Food-101

This notebook demonstrates how to fine-tune your **own custom pre-trained ViT** (trained on CIFAR-100) for a new task (Food-101 Pizza vs Sushi).

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import time

!pip install wandb -q
import wandb

In [18]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Path to your SAVED model from the previous notebook
    PRETRAINED_PATH = '/content/drive/MyDrive/DL/pre_trained_ViT_CIFAR100.pth'
    DATA_PATH = '/content/Food101'
    print("✅ Connected to Google Drive")
except ImportError:
    print("❌ Not running on Colab, using local path")
    PRETRAINED_PATH = "./pre_trained_ViT_CIFAR100.pth" 
    DATA_PATH = "./data"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Log into W&B
wandb.login()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Connected to Google Drive
Using device: cuda


True

## 1. Define the ViT Architecture
This has to be the exact same class definitions as utilized in the pre-training notebook. Any change here will cause weight loading errors.

In [19]:
class EmbeddedPatches(nn.Module):
    def __init__(self, img_size, in_channels, embed_dim, patch_size, batch_size):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, patch_size, patch_size)
        N = (img_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_token = nn.Parameter(torch.randn(1, N+1, embed_dim))

    def forward(self, x):
        B = x.size(0)
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        cls_token = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = x + self.pos_token
        return x

class MLP(nn.Module):
    def __init__(self, in_features, out_features, drop_rate):
        super().__init__()
        self.layer1 = nn.Linear(in_features, out_features)
        self.layer2 = nn.Linear(out_features, in_features)
        self.dropout = nn.Dropout(drop_rate)

    def forward(self, x):
        x = self.layer1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.layer2(x)
        x = self.dropout(x)
        return x

class VisionEncoder(nn.Module):
    def __init__(self, embed_dim, msa_size, mlp_dim, enc_dim, drop_rate):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, msa_size, drop_rate, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = MLP(embed_dim, mlp_dim, drop_rate)

    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ViT(nn.Module):
    def __init__(self, img_size, patch_size, batch_size, in_channels, embed_dim, enc_dim, msa_size, mlp_dim, cls_nums, drop_rate):
        super().__init__()
        self.embed = EmbeddedPatches(img_size, in_channels, embed_dim, patch_size, batch_size)
        self.encoder = nn.Sequential(*[
            VisionEncoder(embed_dim, msa_size, mlp_dim, enc_dim, drop_rate)
            for _ in range(enc_dim)
        ])
        self.mlp_head = nn.Linear(embed_dim, cls_nums)

    def forward(self, x):
        x = self.embed(x)
        x = self.encoder(x)
        cls_token = x[:, 0]
        x = self.mlp_head(cls_token)
        return x

## 2. Hyperparameters & Configuration

-   We increase resolution to **64x64**. The model was trained on 32x32.

In [20]:
# Pre-training Config (MUST MATCH THE SAVED MODEL)
OLD_IMG_SIZE = 32
PATCH_SIZE = 4
IN_CHANNELS = 3
EMBED_DIM = 256
MLP_DIM = 512
ENC_NUMS = 6
MSA_NUMS = 8
OLD_CLS_NUMS = 100

# Fine-tuning Config
NEW_IMG_SIZE = 64  # Let's double the resolution to start
NEW_CLS_NUMS = 2   # Pizza vs Sushi
BATCH_SIZE = 64
EPOCHS = 10
LR = 1e-4
DROP_RATE = 0.1

## 3. Data Loading (Food-101 Subset)

In [21]:
stats = ((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Standard mean/std

train_transform = transforms.Compose([
    transforms.Resize((NEW_IMG_SIZE, NEW_IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

test_transform = transforms.Compose([
    transforms.Resize((NEW_IMG_SIZE, NEW_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

In [22]:
from torch.utils.data import random_split

full_train_dataset = datasets.Food101(root=DATA_PATH, split='train', 
                            download=True, transform=train_transform)

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

full_train_subset, full_val_subset = random_split(full_train_dataset, (train_size, val_size))

full_test_dataset = datasets.Food101(root=DATA_PATH, split='test', 
                            download=True, transform=train_transform)

100%|██████████| 5.00G/5.00G [04:11<00:00, 19.9MB/s]   


KeyboardInterrupt: 

In [17]:
def filter_classes(dataset, target_classes):
    indices = []
    for i, label_idx in enumerate(dataset._labels):
        label_name = dataset.classses[label_idx]
        if label_name in target_classes:
            indices.append(i)
    return indices

target_classes = ['pizza', 'sushi']

train_indices = filter_classes(full_train_subset, target_classes)
val_indices = filter_classes(full_val_subset, target_classes)
test_indices = filter_classes(full_test_dataset, target_classes)

train_subset = Subset(full_train_subset, train_indices)
val_subset = Subset(full_val_subset, val_indices)
test_data = Subset(full_test_dataset, val_indices)

print(f"✅ Filtered Dataset: {len(train_subset)} train images; {len(val_subset)} validation image;" + 
                            f"{len(test_data)} test images")

NameError: name 'full_train_subset' is not defined

In [ ]:
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=4, pin_memory=True)

val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=4, pin_memory=True)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=4, pin_memory=True)

## 4. Interpolation Logic (Custom ViT)

We need to adapt `embed.pos_token` from 32x32 size to 64x64 size.

In [ ]:
def load_custom_vit_weights(model, weights_path, device):
    # 1. Load the saved weights
    state_dict = torch.load(weights_path, map_location=device)
    
    # 2. Get current model's pos_token shape
    new_pos_token = model.embed.pos_token.data
    # .data to get the raw underlying Tensor
    num_patches_new = new_pos_token.shape[1] - 1 
    # -1 because of one token from cls
    
    # 3. Get old pos_token from file (pre-trained model, 64 in this case)
    old_pos_token = state_dict['embed.pos_token']
    
    print(f"Interpolating: {old_pos_token.shape} -> {new_pos_token.shape}")

    if old_pos_token.shape != new_pos_token.shape:
        # Extract segments
        old_cls_token = old_pos_token[:, 0:1, :] # -> (1, 1, 256)
        old_grid_tokens = old_pos_token[:, 1:, :] # -> (1, 64, 256)
        
        # Calculate Grid Sizes
        grid_size_old = int(math.sqrt(old_grid_tokens.shape[1])) # Should be 32/4 = 8
        grid_size_new = int(math.sqrt(num_patches_new))          # Should be 64/4 = 16
        
        embed_dim = old_grid_tokens.shape[-1]
        
        # Convert the position embeddings from a 1D list into a 2D image-like grid
        old_grid_tokens = (old_grid_tokens.permute(0, 2, 1)
                    .reshape(1, embed_dim, grid_size_old, grid_size_old))
        # .premute(): swapping dimension (Batch, Seq, Feat) -> (Batch, Feat, Seq)
        # .reshape(): take the 64 patches and arrange them into their original
        # 2D square shape (1, 256, 64) --> (1, 256, 8, 8)
        
        # Interpolate
        new_grid_tokens = F.interpolate(
            old_grid_tokens, 
            size=(grid_size_new, grid_size_new), 
            mode='bicubic', 
            align_corners=False
        )
        
        # Reshape back
        new_grid_tokens = new_grid_tokens.flatten(2).transpose(1, 2)
        
        # Recombine
        new_pos_token = torch.cat((old_cls_token, new_grid_tokens), dim=1)
        state_dict['embed.pos_token'] = new_pos_token
        
    # 4. Handle Head Mismatch (100 classes -> 2 classes)
    # Depending on how you saved, it might be 'mlp_head.weight' and 'mlp_head.bias'
    # We usually just DROP the head weights and initialize new ones
    if 'mlp_head.weight' in state_dict:
        del state_dict['mlp_head.weight']
        del state_dict['mlp_head.bias']
        print("✅ Dropped old classification head weights")

    # 5. Load weights (strict=False allows missing head keys)
    model.load_state_dict(state_dict, strict=False)
    return model

In [ ]:
# Initialize Model with NEW Configuration (64x64, 2 Classes)
model = ViT(NEW_IMG_SIZE, PATCH_SIZE, BATCH_SIZE, IN_CHANNELS, 
            EMBED_DIM, ENC_NUMS, MSA_NUMS, MLP_DIM, NEW_CLS_NUMS, DROP_RATE).to(device)

# Load and Interpolate weights
model = load_custom_vit_weights(model, PRETRAINED_PATH, device)
print("✅ Custom Model Loaded Successfully")

# Zero-initialize the head like in the ViT Paper
nn.init.zeros_(model.mlp_head.weight)
nn.init.zeros_(model.mlp_head.bias)

## 5. Fine-Tuning Loop

In [ ]:
import numpy as np

class EarlyStopping:
    """Early stops the training if validation loss doesn't improve after a given patience."""
    def __init__(self, patience=7, verbose=False, delta=0, path='checkpoint.pth', trace_func=print):
        """
        Args:
            patience (int): How long to wait after last time validation loss improved.
                            Default: 7
            verbose (bool): If True, prints a message for each validation loss improvement. 
                            Default: False
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
                            Default: 0
            path (str): Path for the checkpoint to be saved to.
                            Default: 'checkpoint.pth'
            trace_func (function): trace print function.
                            Default: print            
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta
        self.path = path
        self.trace_func = trace_func
    
    def __call__(self, val_loss, model):

        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)

        elif score < self.best_score + self.delta:
            self.counter += 1
            self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True

        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        '''Saves model when validation loss decrease.'''
        if self.verbose:
            self.trace_func(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# creates a manager that arificially inflates the gradients during the
# math so they dont disappear into zeros, and then shrinks them back to
# normal size right before the weights are updated.
# ---> x2 faster 
scaler = torch.cuda.amp.GradScaler()
# Mixed Precision torch.cuda.amp: automatically switches to 16-bit numbers (FP16) for
# certain safe operations (Matrix Multiplications) to make training 
# much faster an use half the memory
# During the training, the Gradients often become extremly small,
# especially deep inside a ViT. With FP16, they might turn 0 (Gradient UnderFlow)
# --> The model stops learning completely!
# GradScaler prevents this:
# - Scale up: Before the backward pass, the Scaler multiplies all the losses
# by a large number --> push them out of the danger zone
# - The Backward Pass happens safely using FP16
# - Scale down: Before the optimizer updates the weights, the Scaler
# divides the gradients back down by that exact same large number.

In [ ]:

def train(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pizza_idx = full_train_data.class_to_idx['pizza']
        y = (y != pizza_idx).long()

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():
            out = model(x)
            loss = criterion(out, y)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

In [ ]:
def validate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    pizza_idx = full_train_data.class_to_idx['pizza']
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            y = (y != pizza_idx).long()
            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0

    pizza_idx = full_test_dataset.class_to_idx['pizza']
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            y = (y != pizza_idx).long()

            out = model(x)
            correct += (out.argmax(1) == y).sum().item()

    return correct / len(loader.dataset)

In [ ]:
wandb.init(
    project="Fine-Tuning-ViT-self-pretrained-model",
    config={
        "learning_rate": LR,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMG_SIZE,
        "architecture": "ViT",
        "patch_size": PATCH_SIZE,
        "embed_dim": EMBED_DIM,
        "enc_nums": ENC_NUMS,
        "msa_nums": MSA_NUMS,
        "mlp_dim": MLP_DIM,
        "drop_rate": DROP_RATE
    }
)

train_accuracies, test_accuracies = [], []
early_stopping = EarlyStopping(patience=5, path='FT_best_model.pth')

for epoch in range(EPOCHS):
    start = time.time()

    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    val_loss, val_acc = validate(model, val_loader, criterion)
    test_acc = evaluate(model, test_loader)

    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

    print(f"{epoch}/{EPOCHS}: train_loss: {train_loss:.4f}, val_loss: {val_loss:.4f}," +
                             f" train_acc: {train_acc:.4f}, val_acc: {val_acc:.4f}, test_acc: {test_acc:.4f}")

    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_acc": train_acc,
        "train_loss": train_loss,
        "test_acc": test_acc,
        "lr": optimizer.param_groups[0]['lr']
    })

    early_stopping(val_loss, model)

    if early_stopping.early_stop:
        print("Early Stopping")
        break

model.load_state_dict(torch.load('FT_best_model.pth'))
print("Loaded best model")

wandb.finish()

In [ ]:
import os

model_name = 'FT_ViT_Pizza_Sushi.pth'
save_path = os.path.join(DATA_PATH, model_name)

torch.save(model.stat_dict(), save_path)

print(f"✅ Model saved to: {save_path}")

In [ ]:
plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(test_accuracies, label="Test Accuracy")
plt.ylabel("Accuracy")
plt.xlabel("Test Accuracy")
plt.legend()
plt.title("Train and Test Accuracy")
plt.show()